# Endpoints

An endpoint, in traditional sense, can be a bridge or a communication channel. Here endpoint is bridge used by the Model, to connect to tables, semantics, embeddings or even other Models. 
These endpoints are functions, which execute a given query. 

For example, we embedded our data using `vector embeddings` but now we want to do a semantic search on it. We will use the `Endpoint`. 

In [ ]:
CREATE ENDPOINT similarity_endpoint(query String "Description of the query") AS
WITH tbl AS (
  SELECT CAST(embed($query) AS `Array`(`Float32`)) AS query
) 
SELECT 
	p.id as id,
	p.content as content,
	cosineDistance(embeddings, query) as similarity,
	p.filename as company
FROM 
  pdf_embeddings AS pe 
JOIN 
  pdfs AS p ON p.id = pe.id
CROSS JOIN 
  tbl 
ORDER BY 
  similarity ASC 
  LIMIT 5
WITH description = "Endpoint to get TOP 5 results to the user's query";

What this endpoint is doing is, taking the query, and generating its embeddings. These embeddings are used by `cosineDistance` to get the closest embeddings to user's query. Here closest, means most similar. 

This was just one example of an endpoint. Other examples include function calling, such as for [semantics](../../sql_functions/semantics):

In [ ]:
CREATE ENDPOINT get_semantics() AS 
SELECT * FROM semantics()
WITH description = 'Schemas of all the tables';


Endpoint allows user to create functions, for the model to decide which endpoint to execute for getting the required context for that particular query. 

This also mean, that a model can call for another model using endpoint. For that, we need to wrap a model in an endpoint:

In [ ]:
CREATE PROMPT evaluator_qa(
  system "Given a question and possible answer, respond if it is relevant or not. 
Answer 'yes' if the score is below 0.1. If score is not provided, return 'no'.
Please use 'yes' or 'no' as the answer. 
question: {{question}}
possible answer: {{answer}}
similarity score: {{score}}"
);

In [ ]:
CREATE MODEL evaluator(
  provider 'OpenAI',
  model_name 'gpt-3.5-turbo',
  prompt_name "evaluator_qa",
  execution_options (retries 1),
  args ["question", "answer", "score"]
);

In [ ]:
CREATE ENDPOINT evaluator(question String, answer String, score String) AS
SELECT evaluator($question, $answer, $score);

Here, the model `evaluator` can be executed by another model, which tells if the answer by the model is relevant or not. 